# Files, CSV and JSON
> Read and write files, and convert CSV and JSON

Read text with `•nget`, parse it with `•csv` or `•json`, and calculate. `•tocsv` and `•tojson` produce text, which `•nput` writes to a file.

```text
sales←•csv •nget 'sales.csv'
totals←+/¨sales
'totals.json' •nput •tojson totals
```

Options go on the left as a keyed vector, and the data goes on the right. Option names are case-insensitive.

## Files

In [ ]:
]help •nget

`•nget path` reads a UTF-8 text file.

`X •nget path` takes options on the left: `binary` (`1` reads a vector of byte values) and `encoding` (`'UTF-8'`).

Errors: VALUE for missing files, invalid UTF-8 and other file errors; DOMAIN for invalid options.

In [ ]:
]help •nput

`path •nput data` writes `data` to a new UTF-8 file. It returns the number of bytes written.

`X •nput data` takes options on the left: `path`, `overwrite` (`1` replaces an existing file), `binary` (`1` writes a vector of byte values) and `encoding` (`'UTF-8'`). Plain text on the left is the path.

Errors: VALUE for an existing file without `overwrite`, and for other file errors; DOMAIN for invalid options or byte values, or `binary` with `encoding`; RANK for data that is not a vector.

```apl
text←•nget 'sales.csv'
'copy.csv' •nput text
('path':'copy.csv' ⋄ 'overwrite':1) •nput text
```

Directories must already exist. Binary reads return exact integers in `0…255`. Binary writes accept a vector of integral numbers in that range. They check the data before opening the file.

```apl
bytes←('binary':1) •nget 'image.bin'
('path':'copy.bin' ⋄ 'binary':1) •nput bytes
'UTF-8' •ucs bytes           ⍝ decode UTF-8 when appropriate
bytes-256x×bytes≥128x         ⍝ signed-byte interpretation
```

## Numeric input

In [ ]:
]help •vfi

`•vfi text` returns `(valid ⋄ numbers)` for the whitespace-separated fields of `text`. An invalid field has flag `0x` and value `0`. Fields are parsed as numbers, never executed.

`separators •vfi text` splits on each character in `separators` instead. It trims whitespace around fields. An empty field is a valid `0`.

Errors: DOMAIN when either argument is not text.

Invalid fields have flag `0x` and value `0`. Valid fields have flag `1x`:

In [ ]:
valid nums←•vfi '12 nope -3 1.5'
valid
nums
valid/nums

1ₓ 0ₓ 1ₓ 1ₓ

12 0 ¯3 1.5

12 ¯3 1.5

Numbers retain their literal domains: ordinary spelling is approximate; `x` and `r` are exact. Complex `j`/`J`, infinity, ASCII signs and signed exponents are accepted. `inf`/`infinity` also denote infinity. NaN and invalid numbers produce flag `0x`.

In [ ]:
2⊃•vfi '2 3x 1r4 1j-2 -1e-3 ∞'

2 3ₓ 1r4 1j¯2 ¯0.001 ∞

`separators •vfi text` splits on any supplied character and trims surrounding whitespace. Empty fields are valid zero. Internal whitespace remains part of the field.

In [ ]:
',' •vfi '3.9,2.4,,76,'
'⋄' •vfi '1 ⋄ 2 3 ⋄ 4'

(1ₓ 1ₓ 1ₓ 1ₓ 1ₓ) (3.9 2.4 0 76 0)

(1ₓ 0ₓ 1ₓ) (1 0 4)

Empty input returns two empty vectors. With no separator characters (`''` on the left), nonempty input is one field. DOMAIN: either argument is not character text.

## JSON

In [ ]:
]help •json

`•json text` parses JSON. Objects become keyed vectors. Arrays become vectors. Strings become character vectors. Integers stay exact. `true` and `false` become `1x` and `0x`.

`('fill':v) •json text` replaces `null` with `v`. The default is `∞`.

Errors: DOMAIN for malformed JSON, with its line and column.

Nested structure is kept:

In [ ]:
person←•json '{"name":"Ann","scores":[10,20]}'
'name'⊃person
'scores'⊃person

Ann

10ₓ 20ₓ

In [ ]:
]help •tojson

`•tojson Y` returns JSON text. Keyed axes become objects. Unkeyed axes become arrays. Character vectors become strings. Keyed entries that hold functions, such as `_mime_` renderers, are left out.

`('fill':v) •tojson Y` writes `v` as `null`.

Errors: DOMAIN for infinity without `fill`, out-of-range floats, nonintegral rationals, complex numbers and other functions.

`'f.json' •nput •tojson Y` writes JSON to a file. Scalar arrays export their contents. Axis names and empty-array prototypes are left out:

In [ ]:
•tojson ('name':'Ann' ⋄ 'scores':10x 20x)
•tojson [1x 2x ⋄ 3x 4x]
•tojson •json '{}'

{"name":"Ann","scores":[10,20]}

[[1,2],[3,4]]

{}

Large integers stay exact. Decimal and exponent tokens become floats. `true` and `false` export back as numbers:

In [ ]:
•json '[9223372036854775808,1.5,true,false]'
•tojson •json '[true,false]'

9223372036854775808ₓ 1.5 1ₓ 0ₓ

[1,0]

`fill` replaces `null` on import. On export, the `fill` value becomes `null`:

In [ ]:
•json '[1,null,3]'
('fill':¯1x) •json '[1,null,3]'
('fill':∞) •tojson 1x ∞ 3x

1ₓ ∞ 3ₓ

1ₓ ¯1ₓ 3ₓ

[1,null,3]

Duplicate object members keep the last value.

## CSV

In [ ]:
]help •csv

`•csv text` parses CSV into a vector of columns. Headers become keys. Numeric columns become numbers. Missing numeric cells become `∞`. Missing text cells become `''`.

`X •csv text` takes options on the left: `header`, `separator`, `quotechar`, `doublequote`, `escapechar`, `decimal`, `thousands`, `trim`, `fill`, `text_columns`, `numeric_columns` and `missing`. The Files, CSV and JSON guide describes them.

Errors: DOMAIN for invalid options or duplicate headers; LENGTH for unequal record widths.

Each numeric column uses compact integer or float storage where possible:

In [ ]:
nl←•ucs 10
text←'price,qty',nl,'10.5,2',nl,'20.0,4'
T←•csv text
T
'qty'⊃T
+/¨T

('price':10.5 20 ⋄ 'qty':2ₓ 4ₓ)

2ₓ 4ₓ

('price':30.5 ⋄ 'qty':6ₓ)

In [ ]:
]help •tocsv

`•tocsv T` returns CSV text for a vector of columns. Keys supply the header. Column lengths must agree.

`X •tocsv T` takes options on the left: `header`, `separator`, `quotechar`, `doublequote`, `escapechar`, `decimal`, `thousands`, `trim`, `fill`, `forcequotes` and `lineending`. `fill` writes that exact value as an empty cell. `'forcequotes':2` quotes every field.

Errors: DOMAIN for nonintegral rationals, complex numbers, functions or nested cells; LENGTH for unequal columns.

`'f.csv' •nput •tocsv T` writes CSV to a file. Unkeyed input writes the data without a header. Parsing the text restores the columns:

In [ ]:
T←('price':10.5 20 ⋄ 'qty':2x 4x)
•csv •tocsv T

('price':10.5 20 ⋄ 'qty':2ₓ 4ₓ)

### CSV options

In [ ]:
nl←•ucs 10
text←'price;qty',nl,'10,5;2',nl,'20,0;4'
opts←('separator':';' ⋄ 'decimal':',')
T←opts •csv text
T
csv←opts •tocsv T
opts •csv csv

('price':10.5 20 ⋄ 'qty':2ₓ 4ₓ)

('price':10.5 20 ⋄ 'qty':2ₓ 4ₓ)

| Option | Default | Meaning |
|---|---|---|
| `header` | Import: `1`; export: has keys | Read/write column names |
| `separator` | `','` | Field separator; use `•ucs 9` for TSV |
| `quotechar` | `'"'` | Quote character; `''` disables quoting |
| `doublequote` | `1` | Represent a quote inside a quoted field by doubling it |
| `escapechar` | `''` | Escape character inside quoted fields |
| `decimal` | `'.'` | Decimal mark: `'.'` or `','` |
| `thousands` | `''` | Group separator; groups after the first contain three digits |
| `trim` | `0` | Trim surrounding whitespace from fields |
| `fill` | Import: `∞`; export: none | Numeric missing-cell replacement; on export, this exact value writes as an empty cell |
| `text_columns` | `⍬` | Import these columns as text |
| `numeric_columns` | `⍬` | Import these columns as numbers; invalid nonmissing text errors |
| `missing` | `⍬` | Additional missing-cell strings; empty cells are always missing |
| `forcequotes` | `0` | Export: `0` as needed, `2` all fields |
| `lineending` | `•ucs 10` | Export: LF or CRLF (`•ucs 13 10`) |

Column selectors are names or 1-origin positions. A character vector names one column. Use a vector for several selectors.

In [ ]:
text←'id,qty',(•ucs 10),'00123,5'
T←('text_columns':'id') •csv text
'id'⊃T

(00123)

Separator, quote and escape must be distinct ASCII characters other than CR, LF or NUL. Fields support Unicode, doubled quotes and embedded newlines. Import accepts LF, CRLF and CR record endings. Export with `escapechar` quotes every field and doubles literal escape characters. With quoting disabled, export errors if a field requires quoting.

### Numbers and missing cells

Inference examines a whole column, ignoring missing cells. Integer fields produce exact numbers. Decimal/scientific notation produces floats. A column containing other text stays text. Quoted numbers participate in inference too.

In [ ]:
nl←•ucs 10
T←•csv 'count,label',nl,'2,001',nl,'3,yes'
'count'⊃T
'label'⊃T

2ₓ 3ₓ

(001) (yes)

Integer-only columns use `i64` storage when values fit. Decimal columns promote integers to floats only when every conversion is exact. Other numeric columns retain mixed storage. Integers beyond `i64` remain exact.

Missing numeric cells become `∞`. Missing text cells become `''`. Entirely missing columns are text unless forced numeric.

In [ ]:
nl←•ucs 10
T←•csv 'price,qty',nl,'10.5,2',nl,',4'
'price'⊃T
'qty'⊃T

10.5 ∞

2ₓ 4ₓ

Configure extra missing markers and numeric fill explicitly.

In [ ]:
text←'qty',(•ucs 10),'NA',(•ucs 10),'4'
('missing':'NA' ⋄ 'fill':¯1x) •csv text

('qty':¯1ₓ 4ₓ)

Export uses `-` for negative numbers, decimal/scientific float spelling and plain exact integers. Infinity writes as `inf` or `-inf`. Set `fill` to export a chosen numeric sentinel as an empty field. Nonintegral rationals, complex numbers, functions and nested nonstring cells give DOMAIN.

Empty input returns an empty unkeyed vector. Header-only input returns keyed empty columns. Empty header names are allowed; duplicate names give DOMAIN. Unequal record widths or column lengths give LENGTH. Invalid forced numeric fields report their row and column.